# Itertools => Grouping With groupby

`groupby` splits a stream into runs of items that share a key.

| Term | Meaning |
|---|---|
| `groupby(iterable, key=None)` | Yields `(key, group)` pairs |
| `key` | Function that computes the group value (default: the item itself) |
| `group` | An iterator over the items of one run |
| Consecutive only | Equal keys that are apart form separate groups |
| Sort first | `sorted(data, key=key)` with the **same** key |
| Consume the group | Turn it into a list before moving on |

---

## `groupby(iterable, key=None)`

Yields `(key, group)` pairs, where `group` is an iterator over the items that share that key.

```python
from itertools import groupby

for key, group in groupby("AAABBC"):
    print(key, list(group))
# A ['A', 'A', 'A']
# B ['B', 'B']
# C ['C']
```

- `key` is a function that computes the grouping value. It defaults to the item itself.
- Every time the key **changes**, a new group starts.

## It Groups Consecutive Items Only

`groupby` works like the Unix `uniq` command. It does **not** collect equal keys from different places, unlike SQL `GROUP BY`.

```python
[(k, len(list(g))) for k, g in groupby("AABAA")]
# [('A', 2), ('B', 1), ('A', 2)]     'A' appears in two groups
```

To group all equal keys together, **sort first with the same key function**:

```python
data = sorted(data, key=key_function)
for key, group in groupby(data, key_function):
    ...
```

`sorted()` is stable, so items in each group keep their original order.

## Groups Are Tied to the Source

Each `group` shares the underlying iterator with `groupby`. When `groupby` moves to the next group, the previous one becomes unusable.

```python
groups = [list(g) for k, g in groupby(data, key)]     # correct: consume immediately
list(groupby("AAB"))                                    # groups are already invalid here
```

Always convert a group to a `list` (or process it) **before** advancing.

## Common Patterns

| Pattern | Idea |
|---|---|
| Run-length encoding | `[(k, len(list(g))) for k, g in groupby(text)]` |
| Group records by field | Sort by the field, then `groupby(records, key=itemgetter("field"))` |
| Count items per group | `sum(1 for _ in group)` |
| Consecutive ranges | `groupby(enumerate(nums), key=lambda t: t[1] - t[0])` |
| Remove consecutive duplicates | `[k for k, _ in groupby(items)]` |

## `groupby` vs a Dictionary

| | `groupby` | `defaultdict(list)` |
|---|---|---|
| Needs sorted input | Yes | No |
| Streams data lazily | Yes | No, keeps everything |
| Groups repeated keys | Only consecutive | All |
| Best for | Sorted or naturally clustered data | Unsorted data |

## Key Rules

- **Sort first** with the same key function, unless you want consecutive groups.
- **Consume each group** before moving on.
- The key does not have to be a value from the item. It can be computed.

## Source

https://docs.python.org/3/library/itertools.html#itertools.groupby

In [ ]:
from itertools import groupby
from operator import itemgetter

# Basic groupby: consecutive equal keys
for key, group in groupby("AAABBC"):
    print(key, list(group))

# It groups CONSECUTIVE items only
print([(k, len(list(g))) for k, g in groupby("AABAA")])     # 'A' appears twice

# Sort first to group all equal keys
text = "AABAA"
print([(k, len(list(g))) for k, g in groupby(sorted(text))])

# Run-length encoding
print([(k, len(list(g))) for k, g in groupby("WWWBBWWW")])

# Group records by a field: sort with the SAME key, then group
records = [
    {"city": "Cairo", "name": "Ann"},
    {"city": "Alexandria", "name": "Bob"},
    {"city": "Cairo", "name": "Cy"},
    {"city": "Alexandria", "name": "Dee"},
]
by_city = itemgetter("city")
for city, members in groupby(sorted(records, key=by_city), key=by_city):
    print(city, [m["name"] for m in members])

# Count items per group
print({k: sum(1 for _ in g) for k, g in groupby(sorted("mississippi"))})

# Group by a computed key
words = ["ant", "bee", "cat", "bird", "fish", "eagle"]
for length, group in groupby(sorted(words, key=len), key=len):
    print(length, list(group))

# Consecutive ranges of numbers
nums = [1, 2, 3, 7, 8, 10, 11, 12]
runs = [
    [n for _, n in run]
    for _, run in groupby(enumerate(nums), key=lambda pair: pair[1] - pair[0])
]
print(runs)                                          # [[1, 2, 3], [7, 8], [10, 11, 12]]

# Remove consecutive duplicates
print([k for k, _ in groupby([1, 1, 2, 2, 2, 3, 1, 1])])     # [1, 2, 3, 1]

# The trap: a group is invalid once groupby moves on
pairs = list(groupby("AAB"))
print([(k, list(g)) for k, g in pairs])              # groups are already empty

# The fix: consume each group immediately
print([(k, list(g)) for k, g in groupby("AAB")])